# PersonalAI — Exploratory Data Analysis (MovieLens 25M)

**Every chart in this notebook answers a stated question.** Computation and plotting logic live in `src/data/eda.py`; this notebook orchestrates and interprets.

Prerequisite: run `python scripts/download_data.py` and `python scripts/preprocess.py` first.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

import json
import pandas as pd

from src.data import eda
from src.data.loader import load_processed
from src.utils.logging_config import configure_logging

configure_logging("INFO")
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

## 0. Data quality report

First: what did the cleaning pipeline actually change? (Counts from `data/processed/quality_report.json` — never hand-waved.)

In [ ]:
report = json.loads((ROOT / "data" / "processed" / "quality_report.json").read_text())
print(json.dumps(report, indent=2))

## 1. Load processed data

In [ ]:
movies = load_processed("movies")
ratings = load_processed("ratings")
tags = load_processed("tags")
genome = load_processed("genome_scores")

display(movies.head(3))
display(ratings.head(3))
print(f"movies={len(movies):,}  ratings={len(ratings):,}  "
      f"tags={len(tags):,}  genome_rows(>=0.5)={len(genome):,}")

## 2. Movies

**Q: Which genres dominate the catalog?** (Feature engineering: long-tail genres need different handling than dominant ones; also the diversity stage needs to know what 'diverse' means here.)

In [ ]:
genre_counts = eda.plot_genre_distribution(movies)
genre_counts.tail(10)

**Q: Is the catalog recency-biased?** (Freshness signals must be normalized against catalog composition — a 2020 movie having more ratings than a 1940 movie may just reflect catalog size.)

In [ ]:
decades = eda.plot_release_year_distribution(movies)
decades

## 3. Users and ratings

**Q: Do users round to whole stars?** (Half-star ratings are rarer — matters for implicit-feedback weighting and thresholding 'positive' interactions.)

In [ ]:
rating_counts = eda.plot_rating_distribution(ratings)
rating_counts

**Q: How unequal is user activity?** (CF coverage depends on this: users with 3 ratings are hard to model; the cold-start strategy in Milestone 7 targets them.)

In [ ]:
per_user = eda.plot_user_activity(ratings)
per_user.describe()

**Q: What share of ratings comes from active vs. inactive users?** (Decides whether models trained on all users are dominated by a minority.)

In [ ]:
activity_split = eda.plot_active_vs_inactive(per_user, threshold=50)
activity_split

## 4. Relationships

**Q: Do widely-rated movies also get rated higher?** (Informs the Bayesian-smoothed popularity score — and the answer 'yes, weakly' is itself a finding worth writing up.)

In [ ]:
movie_stats = ratings.groupby("movieId").agg(
    mean_rating=("rating", "mean"), n_ratings=("rating", "size")
)
eda.plot_popularity_vs_rating(movie_stats)
correlation = movie_stats["mean_rating"].corr(movie_stats["n_ratings"])
print(f"Pearson r(mean_rating, n_ratings) = {correlation:+.3f}")

**Q: Which genres get the highest mean ratings?** (The user preference model will represent users as genre distributions — we need to know the global baseline they deviate from.)

In [ ]:
genre_means = eda.plot_genre_vs_rating(ratings, movies)
genre_means

**Q: Does movie age predict attention?** (The freshness signal needs a factual basis: is 'newer = more rated' actually true in this catalog?)

In [ ]:
decade_medians = eda.plot_year_vs_popularity(ratings, movies)
decade_medians

## 5. Findings summary

(Completed after running — real numbers only. Template:)

| # | Question | Finding | Consequence for the system |
|---|---|---|---|
| 1 | Catalog genres | *fill from results* | |
| 2 | Recency bias | *fill* | |
| 3 | Rating rounding | *fill* | |
| 4 | Activity inequality | *fill* | |
| 5 | Active/inactive split | *fill* | |
| 6 | Popularity↔rating | *fill* | |
| 7 | Genre rating baseline | *fill* | |
| 8 | Age↔attention | *fill* | |